# 如何添加消息历史

`RunnableWithMessageHistory` 类允许我们为某些类型的链添加消息历史

该类还通过使用 `session_id` 保存每个对话来支持多个对话 - 然后在调用可运行对象时期望在配置中传递 `session_id`，并使用它查找相关的对话历史

![alt text](message_history-4c13b8b9363beb4621d605bf6b5a34b4.png)

```py
from langchain_core.runnables.history import RunnableWithMessageHistory


with_message_history = RunnableWithMessageHistory(
    # The underlying runnable
    runnable,  
    # A function that takes in a session id and returns a memory object
    get_session_history,  
    # Other parameters that may be needed to align the inputs/outputs
    # of the Runnable with the memory object
    ...  
)

with_message_history.invoke(
    # The same input as before
    {"ability": "math", "input": "What does cosine mean?"},
    # Configuration specifying the `session_id`,
    # which controls which conversation to load
    config={"configurable": {"session_id": "abc123"}},
)
```

 在构造 `RunnableWithMessageHistory` 时，你需要传入一个 `get_session_history` 函数。 
 
 - get_session_history (session_id) ---> BaseChatMessageHistory (这些类通常使用会话 ID 初始化。)

- RunnableWithMessageHistory 只能包装某些类型的可运行对象。具体来说，它可以用于任何输入为以下之一的可运行对象：

    - 一系列 BaseMessages
    - 一个字典，键为一系列 BaseMessages
    - 一个字典，键为最新的消息（作为字符串或 BaseMessages 的序列），另一个键为历史消息
    
- 并返回作为输出之一

    - 可以被视为 AIMessage 内容的字符串
    - 一系列 BaseMessage
    - 一个字典，键包含一系列 BaseMessage

In [1]:
import os
from dotenv import load_dotenv
from langchain_deepseek import ChatDeepSeek

load_dotenv("apikey.env")
BASE_URL = 'https://api.deepseek.com'
API_KEY = os.getenv('DEEPSEEK-API-KEY')
deepseek_chat_model = 'deepseek-chat'
if  not API_KEY:
    raise ValueError("WARNING: NOT FOUND OPENAI_API_KEY，PLEASE CHECK .env SETING。")
else:
    print("SECESSFULLY!")
model = ChatDeepSeek(api_key=API_KEY, base_url=BASE_URL, model=deepseek_chat_model)

SECESSFULLY!


## 消息输入，消息输出
最简单的形式就是给 ChatModel 添加内存。

In [2]:
from langchain_community.chat_message_histories import SQLChatMessageHistory


def get_session_history(session_id):
    return SQLChatMessageHistory(session_id, "sqlite:///memory.db")

In [3]:
from langchain_core.messages import HumanMessage
from langchain_core.runnables.history import RunnableWithMessageHistory

runnable_with_history = RunnableWithMessageHistory(
    model,
    get_session_history,
)

In [4]:
for chunk in runnable_with_history.stream(
    [HumanMessage(content="hi - im bob!")],
    config={"configurable": {"session_id": "1"}},
):
    print(chunk.content, end="", flush=True)

C:\Users\hhm18\miniconda3\envs\TrainingCamp\lib\site-packages\langchain_core\runnables\history.py:598: LangChainDeprecationWarning: `connection_string` was deprecated in LangChain 0.2.2 and will be removed in 1.0. Use connection instead.
  message_history = self.get_session_history(


Hi Bob! 😊 Great to see you again! What can I help you with today?

In [5]:
###########通过上下下文提供的 session_id 聊天历史，模型得以直到用户的名字 ##############

for chunk in runnable_with_history.stream(
    [HumanMessage(content="whats my name?")],
    config={"configurable": {"session_id": "1"}},
):
    print(chunk.content, end="", flush=True)

Your name is **Bob**! 😊 You've told me a few times — it's nice to chat with you again!

In [7]:
###########传递一个不同的 session_id 时，我们开始一个新的聊天历史，模型得不知道用户的名字 ##############
for chunk in runnable_with_history.stream(
    [HumanMessage(content="whats my name?")],
    config={"configurable": {"session_id": "11"}},
):
    print(chunk.content, end="", flush=True)

I don't have access to your name unless you tell me! 😊 If you'd like to share your name, I'd be happy to use it in our conversation. Otherwise, I can just refer to you as "you" - whatever works best for you!

## 字典输入，消息输出
除了简单地包装一个原始模型，下一步是包装一个提示 + 大型语言模型。这现在将输入更改为 字典：
1. 一个字典可以有多个键，但我们只想保存一个作为输入，需要指定一个键来保存为输入。
2. 一旦我们加载了消息，我们需要知道如何将它们保存到字典中。这相当于知道在字典中保存它们的哪个键。因此，我们需要指定一个键来保存加载的消息。

In [17]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You're an assistant who speaks in {language}. Using {language} to respond in 40 words or fewer",
        ),
        MessagesPlaceholder(variable_name="history"), ######
        ("human", "{input}"),
    ]
)

runnable = prompt | model

runnable_with_history = RunnableWithMessageHistory(
    runnable,
    get_session_history,
    input_messages_key="input",  ##被视为最新输入消息的键
    history_messages_key="history",  ##用于添加历史消息的键
)

In [18]:
for chunk in runnable_with_history.stream(
     {"language": "english", "input": "你好，我是飞天小女警😀！"},
    config={"configurable": {"session_id": "2"}},
):
    print(chunk.content, end="", flush=True)

Greetings, Powerpuff Girl! What crime-fighting adventure can we tackle today? 😊

In [19]:
for chunk in runnable_with_history.stream(
     {"language": "italian", "input": "你应该知道我我是谁！"},
    config={"configurable": {"session_id": "2"}},
):
    print(chunk.content, end="", flush=True)

Ah, certo! Sei la famosa Powerpuff Girl! Come posso aiutarti oggi, supereroina? 😊

## 消息输入，字典输出


In [26]:
from langchain_core.messages import HumanMessage
from langchain_core.runnables import RunnableParallel

chain = RunnableParallel({"output_message": model})


runnable_with_history = RunnableWithMessageHistory(
    chain,
    get_session_history,
    output_messages_key="output_message",
)

runnable_with_history.invoke(
    [HumanMessage("你好，我是鲨鱼辣椒！")],
    config={"configurable": {"session_id": "3"}},
)

{'output_message': AIMessage(content='（触角天线嗡嗡发光）**警告！第三次身份声明触发隐藏协议！**  \n启动【鲨鱼辣椒特别会话模组】——  \n🔹 选项1：用“友谊头槌”切磋打招呼  \n🔹 选项2：联名举报蜻蜓队长私藏和平星  \n🔹 选项3：共享今日反派能量补充计划（冰淇淋优先）  \n请下达指令！🗿🚁', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 89, 'prompt_tokens': 166, 'total_tokens': 255, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 128}, 'prompt_cache_hit_tokens': 128, 'prompt_cache_miss_tokens': 38}, 'model_name': 'deepseek-chat', 'system_fingerprint': 'fp_ffc7281d48_prod0820_fp8_kvcache', 'id': 'f47c9341-05d4-4188-b808-afea79caf4fb', 'service_tier': None, 'finish_reason': 'stop', 'logprobs': None}, id='run--4b963fde-f6d8-4395-a47f-bc99ed1dbbbb-0', usage_metadata={'input_tokens': 166, 'output_tokens': 89, 'total_tokens': 255, 'input_token_details': {'cache_read': 128}, 'output_token_details': {}})}

In [27]:
runnable_with_history.invoke(
    [HumanMessage("你还记得我吗？")],
    config={"configurable": {"session_id": "3"}},
)

{'output_message': AIMessage(content='（装甲接缝发出轻微的充能声）**核心记忆库检索中——**  \n⚠️检测到【鲨鱼辣椒特殊频段】！怎么可能忘记？  \n您可是在《铁甲小宝》片场用**鲨鱼巨人**和我方卡布达小队抢过和平星的传奇！（虽然最后总被忽悠着一起救地球…）  \n需要启动了【怀旧联机模式】还是开启【新任务存档】？🗂️🤖', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 98, 'prompt_tokens': 263, 'total_tokens': 361, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 192}, 'prompt_cache_hit_tokens': 192, 'prompt_cache_miss_tokens': 71}, 'model_name': 'deepseek-chat', 'system_fingerprint': 'fp_ffc7281d48_prod0820_fp8_kvcache', 'id': 'c632f7b2-1372-46fd-b0e9-ebf237e3450b', 'service_tier': None, 'finish_reason': 'stop', 'logprobs': None}, id='run--8adf684d-53a7-4f3e-8bed-490f79dfa34d-0', usage_metadata={'input_tokens': 263, 'output_tokens': 98, 'total_tokens': 361, 'input_token_details': {'cache_read': 192}, 'output_token_details': {}})}

## 单键字典用于所有消息输入和消息输出

这是“字典输入，消息输出”的特定情况。在这种情况下，由于只有一个单键，我们只需要指定 `input_messages_key`

In [29]:
from operator import itemgetter

runnable_with_history = RunnableWithMessageHistory(
    itemgetter("input_messages") | model,
    get_session_history,
    input_messages_key="input_messages",
)

for chunck in runnable_with_history.stream(
    {"input_messages": [HumanMessage(content="任何邪恶😈终将绳之以法！！")]},
    config={"configurable": {"session_id": "4"}},
):
    print(chunck.content, end="", flush=True)

（同步握拳目射金光）说得好！此刻我身后仿佛已响起《铠甲勇士》BGM——  
**刑天铠甲·合体！**  
您的正义宣言已启动全网邪恶扫描程序：  
【检测到😈】→ 启动表情包法槌💥→【罪恶值归零】  
温馨提示：本台刚收到前方战报——邪恶势力正因您的气势连夜转行做奶茶店员🧋（毕竟甜蜜才是终极救赎啊！）

In [30]:
for chunck in runnable_with_history.stream(
    {"input_messages": [HumanMessage(content="要求一字不拉的对暗号：______终将绳之以法！！")]},
    config={"configurable": {"session_id": "4"}},
):
    print(chunck.content, end="", flush=True)

（立正敬礼）报告！暗号完整接收——  
**“任何邪恶😈终将绳之以法！！”**  
（已启动声纹核验程序：✅ 字字精准！情绪浓度500%！  
——正在为您连接正义联盟热线📞——  
⚠️注意：重复该暗号三次将触发😈强制尬舞刑罚）

## 自定义
我们通过传递一组 ConfigurableFieldSpec 对象到 history_factory_config 参数来定制跟踪消息历史的配置参数。